# 04 — Bayesian Smoothing & Recency Penalty

**Goal:** Regularize RDS scores with Bayesian shrinkage and add recency penalties to prevent notification fatigue.

This notebook covers two layers:
- **Bayesian smoothing** — pull noisy scores toward the mean (protects against small-sample luck)
- **Recency penalty** — penalize templates recently sent to the same user (prevents repetition)

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_sample
from src.scoring.difference_score import (
    compute_template_reward_rates,
    compute_relative_difference_scores_fast,
)
from src.scoring.bayesian_smoothing import bayesian_smooth, compute_global_mean
from src.recency.recency_penalty import (
    compute_recency_penalty,
    adjust_scores_with_recency,
    explain_recency_adjustment,
)

sns.set_style("whitegrid")
print("Ready!")

---
## 1. Recompute RDS Scores

We need the raw RDS scores and counts from the previous notebook as our starting point.

In [ ]:
train_df = load_sample(n_rows=100_000, split="train")
reward_rates, counts = compute_template_reward_rates(train_df)
rds_scores = compute_relative_difference_scores_fast(train_df, reward_rates)

templates = sorted(rds_scores.keys())
print(f"\nReady — {len(templates)} templates with RDS scores.")

---
## 2. The Noise Problem

Not all RDS scores are equally trustworthy. A template sent 10 million times has a very reliable score. A template sent only 50 times? Its score could be wildly off just by luck.

Let's see the count spread:

In [ ]:
count_df = pd.DataFrame({
    "template": templates,
    "rds": [rds_scores[t] for t in templates],
    "count": [counts[t] for t in templates],
})

print("Template observation counts:\n")
print(count_df.sort_values("count").to_string(index=False))
print(f"\nRange: {count_df['count'].min():,} – {count_df['count'].max():,}")
print(f"Templates with few counts have noisier RDS — we shouldn't trust them as much.")

---
## 3. Bayesian Shrinkage

The fix: **pull every score toward the global average**. Templates with lots of data barely move. Templates with little data get pulled strongly.

$$\text{smoothed}(a) = \frac{n_a \times \text{RDS}(a) + \kappa \times \mu}{n_a + \kappa}$$

Where:
- $n_a$ = number of observations for template $a$
- $\kappa$ = shrinkage strength (hyperparameter)
- $\mu$ = global weighted mean of all RDS scores

**Intuition:**
- If $n_a \gg \kappa$: data dominates → score barely changes
- If $n_a \ll \kappa$: prior dominates → score pulled to mean
- When $n_a = \kappa$: score is halfway between data and prior

In [ ]:
kappa = 1000
smoothed_scores = bayesian_smooth(rds_scores, counts, kappa)

In [ ]:
# Before vs After chart
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(templates))
width = 0.35

raw_vals = [rds_scores[t] for t in templates]
smooth_vals = [smoothed_scores[t] for t in templates]

ax.bar(x - width/2, raw_vals, width, label="Raw RDS", color="#3498db", edgecolor="white")
ax.bar(x + width/2, smooth_vals, width, label=f"Smoothed (κ={kappa})", color="#e67e22", edgecolor="white")
ax.axhline(y=0, color="black", linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(templates)
ax.set_title("Raw RDS vs Bayesian-Smoothed Scores", fontsize=14, fontweight="bold")
ax.set_xlabel("Template")
ax.set_ylabel("Score")
ax.legend()
plt.tight_layout()
plt.show()

print("Templates most affected by smoothing (biggest shift):")
shifts = [(t, abs(smoothed_scores[t] - rds_scores[t]), counts[t]) for t in templates]
shifts.sort(key=lambda x: x[1], reverse=True)
for t, shift, n in shifts[:5]:
    print(f"  {t}: shifted by {shift:.6f}  (n={n:,})")

---
## 4. Effect of κ — How Much Shrinkage?

Higher κ = more conservative (scores pulled more strongly toward the mean).

In [ ]:
kappa_values = [100, 500, 1000, 5000]

results = {}
for k in kappa_values:
    results[k] = bayesian_smooth(rds_scores, counts, k)

# Table
print(f"\n{'Template':<10}", end="")
print(f"{'Raw RDS':>12}", end="")
for k in kappa_values:
    print(f"{'κ='+str(k):>12}", end="")
print()
print("-" * (10 + 12 + 12 * len(kappa_values)))

for t in templates:
    print(f"{t:<10}", end="")
    print(f"{rds_scores[t]:>+12.6f}", end="")
    for k in kappa_values:
        print(f"{results[k][t]:>+12.6f}", end="")
    print()

In [ ]:
# Line chart showing how scores converge as kappa increases
fig, ax = plt.subplots(figsize=(10, 5))

kappas_fine = [50, 100, 200, 500, 1000, 2000, 5000, 10000]
global_mean = compute_global_mean(rds_scores, counts)

for t in templates:
    scores_over_k = []
    for k in kappas_fine:
        s = (counts[t] * rds_scores[t] + k * global_mean) / (counts[t] + k)
        scores_over_k.append(s)
    ax.plot(kappas_fine, scores_over_k, marker="o", markersize=3, label=t)

ax.axhline(y=global_mean, color="black", linestyle="--", alpha=0.5, label="global mean")
ax.set_xscale("log")
ax.set_xlabel("κ (shrinkage strength, log scale)")
ax.set_ylabel("Smoothed Score")
ax.set_title("How Scores Converge Toward the Mean as κ Increases", fontsize=13, fontweight="bold")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Recency Penalty — Concept

Even the best template gets stale if sent repeatedly. The **recency penalty** subtracts a value from a template's score based on how recently it was sent to this user:

$$\text{penalty}(a) = \gamma \times e^{-h \times d}$$

Where:
- $\gamma$ = maximum penalty (when template was just sent, $d = 0$)
- $h$ = decay rate (how fast the penalty fades)
- $d$ = days since template was last sent

The template "recovers" over time — hence the name **Recovering Bandit**.

In [ ]:
# Visualize the decay curve
days = np.linspace(0, 10, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: vary gamma
h_fixed = 0.5
for gamma in [0.05, 0.1, 0.2, 0.3]:
    penalty = gamma * np.exp(-h_fixed * days)
    axes[0].plot(days, penalty, label=f"γ={gamma}")
axes[0].set_title(f"Penalty Decay (varying γ, h={h_fixed})", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Days since last sent")
axes[0].set_ylabel("Penalty")
axes[0].legend()

# Right: vary h
gamma_fixed = 0.1
for h in [0.2, 0.5, 1.0, 2.0]:
    penalty = gamma_fixed * np.exp(-h * days)
    axes[1].plot(days, penalty, label=f"h={h}")
axes[1].set_title(f"Penalty Decay (varying h, γ={gamma_fixed})", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Days since last sent")
axes[1].set_ylabel("Penalty")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Left: higher γ = stronger maximum penalty (bigger hit right after sending)")
print("Right: higher h = faster decay (penalty fades more quickly)")

---
## 6. Worked Example — Recency Adjustment

Let's pick a few templates and simulate a user history to see how scores change.

In [ ]:
gamma = 0.1
h = 0.5

# Use the first 4 templates
demo_templates = templates[:4]
demo_scores = {t: smoothed_scores[t] for t in demo_templates}

print("=" * 55)
print("SCENARIO 1: New user (no history)")
print("Expectation: no penalties, scores unchanged")
print("=" * 55)
adjusted = explain_recency_adjustment(demo_scores, [], gamma, h)

In [ ]:
print("=" * 55)
print(f"SCENARIO 2: Template {demo_templates[0]} sent 0.5 days ago")
print(f"Expectation: {demo_templates[0]} gets penalized, others unchanged")
print("=" * 55)

history = [(demo_templates[0], 0.5)]
adjusted = explain_recency_adjustment(demo_scores, history, gamma, h)

p = compute_recency_penalty(demo_templates[0], history, gamma, h)
print(f"\nPenalty = γ × exp(-h × d) = {gamma} × exp(-{h} × 0.5) = {p:.6f}")

In [ ]:
print("=" * 55)
print("SCENARIO 3: Multiple templates in history")
print("=" * 55)

history_multi = [
    (demo_templates[0], 0.3),   # sent very recently
    (demo_templates[1], 2.0),   # sent 2 days ago
    (demo_templates[2], 7.0),   # sent a week ago
]
print(f"History: {history_multi}")
adjusted = explain_recency_adjustment(demo_scores, history_multi, gamma, h)

print(f"\nNotice how penalty decreases with time:")
print(f"  {demo_templates[0]} (0.3d ago): strong penalty")
print(f"  {demo_templates[1]} (2.0d ago): moderate penalty")
print(f"  {demo_templates[2]} (7.0d ago): almost no penalty")
print(f"  {demo_templates[3]}: not in history → zero penalty")

In [ ]:
# Visualize the three scenarios
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

scenarios = [
    ("No history", []),
    (f"{demo_templates[0]} sent 0.5d ago", [(demo_templates[0], 0.5)]),
    ("3 templates in history", history_multi),
]

for ax, (title, hist) in zip(axes, scenarios):
    adj = adjust_scores_with_recency(demo_scores, hist, gamma, h)
    
    x_pos = np.arange(len(demo_templates))
    w = 0.35
    
    ax.bar(x_pos - w/2, [demo_scores[t] for t in demo_templates], w,
           label="Before", color="#3498db", edgecolor="white")
    ax.bar(x_pos + w/2, [adj[t] for t in demo_templates], w,
           label="After", color="#e74c3c", edgecolor="white")
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(demo_templates)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Score")
plt.suptitle("Recency Penalty Effect on Scores", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Summary

| Layer | What it does | Key parameter |
|-------|-------------|---------------|
| **Bayesian smoothing** | Pulls noisy scores toward global mean | κ (kappa) — higher = more conservative |
| **Recency penalty** | Penalizes recently-sent templates | γ (magnitude), h (decay rate) |

After these two layers, we have **adjusted scores** for each template, personalized to each user's history. The final step is to convert these scores into a probability distribution for template selection.

**Next notebook:** `05_softmax_selection.ipynb` — use softmax to select templates while balancing exploration and exploitation.